# ⚡ EnerVision AI — PatchTSMixer Model

**Standalone Notebook**

This notebook trains and evaluates a single deep learning time-series forecasting model, **PatchTSMixer**, as part of the **EnerVision AI** household energy-forecasting project.
It is designed to run independently in a fresh Google Colab session, with GPU recommended for faster training.

| | |
|---|---|
| **Project** | EnerVision AI |
| **Model** | PatchTSMixer |
| **Task** | Household energy-consumption forecasting |
| **Dataset** | `HomeC_Cleaned.csv` |
| **Environment** | Google Colab (GPU recommended) |

---


## 🔧 Step 1: Install Compatible Library Versions

> **Important:** After running this cell, go to **Runtime → Restart session** in Google Colab, then continue with the following cells.


In [ ]:
!pip uninstall -y numpy scipy transformers accelerate
!pip install "numpy<2.0" "scipy<1.14.0" "transformers>=4.38.0" accelerate


## 📦 Step 2: Import Required Libraries


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from transformers import PatchTSMixerConfig, PatchTSMixerForPrediction
from sklearn.preprocessing import StandardScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## 📂 Step 3: Mount Google Drive and Load the Cleaned Dataset


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

data_path = '/content/drive/MyDrive/EnerVision_AI/Data/HomecCleaned.csv'

print("Loading the cleaned dataset...")
df = pd.read_csv(data_path, low_memory=False)

print(f"✅ Data loaded successfully! Data size: {df.shape}")
display(df.head())


## 🧩 Step 4: Prepare Time-Series Windows (Train/Validation/Test Split)


In [ ]:
target_column = 'use [kW]'  # household consumption/Demand — matches the
                             # unified target now used by XGBoost & Bi-LSTM

if target_column not in df.columns:
    raise ValueError(
        f"The target column '{target_column}' was not found in the dataset."
    )

df['time'] = pd.to_datetime(df['time'])

WEATHER_COLS = ['temperature', 'humidity', 'visibility', 'apparentTemperature',
                'pressure', 'windSpeed', 'cloudCover', 'windBearing',
                'precipIntensity', 'dewPoint', 'precipProbability']
WEATHER_COLS = [c for c in WEATHER_COLS if c in df.columns]

agg_dict = {target_column: 'mean'}
agg_dict.update({c: 'mean' for c in WEATHER_COLS})

df_hourly = (
    df.set_index('time')
      .resample('h')
      .agg(agg_dict)
      .dropna()
      .reset_index()
)

df_hourly['is_weekend'] = (df_hourly['time'].dt.dayofweek >= 5).astype(float)
df_hourly['hour_sin'] = np.sin(2 * np.pi * df_hourly['time'].dt.hour / 24.0)
df_hourly['hour_cos'] = np.cos(2 * np.pi * df_hourly['time'].dt.hour / 24.0)
df_hourly['month_sin'] = np.sin(2 * np.pi * df_hourly['time'].dt.month / 12.0)
df_hourly['month_cos'] = np.cos(2 * np.pi * df_hourly['time'].dt.month / 12.0)
CALENDAR_COLS = ['is_weekend', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos']

# The target MUST be channel 0 — later cells rely on this to slice the
# target back out of the model's multivariate output.
FEATURE_COLS = [target_column] + WEATHER_COLS + CALENDAR_COLS
TARGET_CHANNEL_IDX = 0

print(f"Resampled {df.shape[0]} minute-level rows -> {df_hourly.shape[0]} hourly points.")
print(f"Channels ({len(FEATURE_COLS)}): {FEATURE_COLS}")

data_values = df_hourly[FEATURE_COLS].values  # shape: (n_hours, n_channels)

CONTEXT_LENGTH = 96   # last 96 HOURS (4 days) of history
PREDICTION_LENGTH = 24  # next 24 HOURS — the genuine 24-hour-ahead forecast

# FIX (data leakage): split the RAW, unscaled series chronologically
# FIRST. The scaler must only ever see the training portion.

n = len(data_values)
TEST_FRACTION = 0.2
VAL_FRACTION_OF_TRAINVAL = 0.15  # taken from the train+val portion

train_val_size = int(n * (1 - TEST_FRACTION))
raw_train_val = data_values[:train_val_size]
raw_test = data_values[train_val_size:]

val_size = int(len(raw_train_val) * VAL_FRACTION_OF_TRAINVAL)
raw_train = raw_train_val[:-val_size]
raw_val = raw_train_val[-val_size:]

print("Preparing time-series windows...")
print(f"Raw split sizes -> train: {raw_train.shape}, val: {raw_val.shape}, test: {raw_test.shape}")

scaler = StandardScaler()
scaler.fit(raw_train)

scaled_train = scaler.transform(raw_train)
scaled_val = scaler.transform(raw_val)
scaled_test = scaler.transform(raw_test)


def create_windows(data, context_length, prediction_length):
    X, y = [], []

    for i in range(len(data) - context_length - prediction_length + 1):
        X.append(data[i : i + context_length])

        y.append(
            data[
                i + context_length :
                i + context_length + prediction_length
            ]
        )

    return np.array(X), np.array(y)


# Build windows WITHIN each split separately, so no window ever spans
# a train/val/test boundary (this also avoids temporal leakage).
X_train, y_train = create_windows(scaled_train, CONTEXT_LENGTH, PREDICTION_LENGTH)
X_val, y_val = create_windows(scaled_val, CONTEXT_LENGTH, PREDICTION_LENGTH)
X_test, y_test = create_windows(scaled_test, CONTEXT_LENGTH, PREDICTION_LENGTH)


class TimeSeriesDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_dataset = TimeSeriesDataset(X_train, y_train)
val_dataset = TimeSeriesDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

print(f"✅ Train windows: {X_train.shape} | Validation windows: {X_val.shape} | Test windows: {X_test.shape}")


## 🏗️ Step 5: Configure and Build the PatchTSMixer Model


In [ ]:
config = PatchTSMixerConfig(
    context_length=CONTEXT_LENGTH,
    prediction_length=PREDICTION_LENGTH,
    num_input_channels=len(FEATURE_COLS),  # weather + calendar + target as INPUT
    prediction_channel_indices=[TARGET_CHANNEL_IDX],
    patch_length=16,
    patch_stride=8,
    d_model=64,
    num_layers=3,
    dropout=0.2,
)

model = PatchTSMixerForPrediction(config).to(device)

print(f"✨ PatchTSMixer model architecture created successfully with {len(FEATURE_COLS)} input channels, "
      f"predicting channel {TARGET_CHANNEL_IDX} ('{FEATURE_COLS[TARGET_CHANNEL_IDX]}') only!")


## 🚀 Step 6: Train the Model (Training Loop with Early Stopping)

Instead of a fixed number of epochs, we train up to `MAX_EPOCHS` and monitor the **validation loss** each epoch. Training stops automatically once it stops improving for `PATIENCE` epochs in a row, and the best-performing weights are restored.


In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3
)

MAX_EPOCHS = 100
PATIENCE = 5
PRINT_EVERY = 50  # print progress every N batches within an epoch

best_val_loss = float('inf')
epochs_no_improve = 0
best_state = None
train_losses, val_losses = [], []

print(f"🚀 Starting training | Train batches/epoch: {len(train_loader)} | Val batches: {len(val_loader)}", flush=True)

for epoch in range(MAX_EPOCHS):

    # --- Train ---
    model.train()
    total_loss = 0
    for batch_idx, (batch_x, batch_y) in enumerate(train_loader):
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        outputs = model(past_values=batch_x, future_values=batch_y)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if (batch_idx + 1) % PRINT_EVERY == 0:
            running_avg = total_loss / (batch_idx + 1)
            print(f"  Epoch {epoch + 1} | batch {batch_idx + 1}/{len(train_loader)} | running train loss: {running_avg:.4f}", flush=True)

    train_loss = total_loss / len(train_loader)

    # --- Validate ---
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            outputs = model(past_values=batch_x, future_values=batch_y)
            val_loss += outputs.loss.item()
    val_loss /= len(val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(
        f"Epoch [{epoch + 1}/{MAX_EPOCHS}] - "
        f"Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f}",
        flush=True
    )

    # --- Early stopping ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"⏹️ Early stopping triggered at epoch {epoch + 1} (no improvement for {PATIENCE} epochs).", flush=True)
            break

# Restore the best-performing weights before evaluating/saving
model.load_state_dict(best_state)
print(f"✅ Training complete. Best validation loss: {best_val_loss:.4f} (best weights restored).", flush=True)


In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(train_losses) + 1), train_losses, marker='o', label='Train Loss')
plt.plot(range(1, len(val_losses) + 1), val_losses, marker='o', label='Validation Loss')
plt.title('PatchTSMixer Training & Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## 📊 Step 7: Evaluate PatchTSMixer Performance


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Prepare the test data (Test DataLoader)
test_dataset = TimeSeriesDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

model.eval()
preds_list = []
actuals_list = []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)
        outputs = model(past_values=batch_x)

        preds_list.append(outputs.prediction_outputs.cpu().numpy()) # Shape (batch_size, PREDICTION_LENGTH, 1)
        actuals_list.append(batch_y.numpy()) # Shape (batch_size, PREDICTION_LENGTH, len(FEATURE_COLS))

# Combine predictions and actual values
preds_array = np.concatenate(preds_list, axis=0)    # Shape (n_test, PREDICTION_LENGTH, 1)
actuals_array = np.concatenate(actuals_list, axis=0) # Shape (n_test, PREDICTION_LENGTH, len(FEATURE_COLS))

# Get the number of test samples and prediction length
n_test, pred_len, _ = preds_array.shape # _ will be 1, the number of channels the model predicted

# The full number of features the scaler was trained on
num_total_features = len(FEATURE_COLS) # This is 17

# Inverse transform actuals (which has all 'num_total_features' channels)
# Reshape actuals_array for scaler: (total_samples, num_total_features)
actuals_2d_for_scaler = actuals_array.reshape(-1, num_total_features) # Shape (n_test * pred_len, num_total_features)
actuals_inv_full = scaler.inverse_transform(actuals_2d_for_scaler)
actuals_inv = actuals_inv_full.reshape(n_test, pred_len, num_total_features)

# Inverse transform predictions (which only has 1 channel - the target)
# preds_array has shape (n_test, PREDICTION_LENGTH, 1)
# Extract scaler's mean and scale for the TARGET_CHANNEL_IDX
mean_target_channel = scaler.mean_[TARGET_CHANNEL_IDX]
scale_target_channel = scaler.scale_[TARGET_CHANNEL_IDX]

# Manually apply inverse transform to the predictions, then remove the last dimension (size 1)
# preds_array is already scaled, so we apply the inverse transform for the target channel
preds_target = (preds_array * scale_target_channel + mean_target_channel).squeeze(axis=-1) # Shape (n_test, PREDICTION_LENGTH)

# Extract the target channel from the inverse transformed actuals
actuals_target = actuals_inv[:, :, TARGET_CHANNEL_IDX] # Shape (n_test, PREDICTION_LENGTH)

# Overall metrics across all 24 forecasted hours combined
mae = mean_absolute_error(actuals_target.flatten(), preds_target.flatten())
mse = mean_squared_error(actuals_target.flatten(), preds_target.flatten())
rmse = np.sqrt(mse)
r2 = r2_score(actuals_target.flatten(), preds_target.flatten())

print("\n" + "=" * 30)
print("📊 PatchTSMixer Model Performance Evaluation (full 24-hour horizon):")
print(f"• RMSE (Root Mean Squared Error): {rmse:.4f}")
print(f"• MAE (Mean Absolute Error): {mae:.4f}")
print(f"• R2 Score (Coefficient of Determination): {r2:.4f}")
print("=" * 30 + "\n")

# Per-step breakdown (accuracy 1 hour ahead vs. 24 hours ahead) — useful
# for the report to show how accuracy degrades further into the horizon.
per_step_mae = np.abs(actuals_target - preds_target).mean(axis=0)
print("MAE by forecast step (hours ahead):")
for h, v in enumerate(per_step_mae, start=1):
    print(f"  +{h:2d}h: {v:.4f} kW")

# Kept for compatibility with the "Actual vs Predicted" plot cell below,
# which expects 2D [:, 0]-indexable arrays for the first forecasted hour.
# Both preds_target and actuals_target are (n_test, PREDICTION_LENGTH)
preds_flat = preds_target[:, 0:1] # Selects the first column, keeps 2D shape (n_test, 1)
actuals_flat = actuals_target[:, 0:1] # Selects the first column, keeps 2D shape (n_test, 1)


## 📝 Step 8: Save Metrics for Model Comparison


In [ ]:
import os

RESULTS_PATH = '/content/drive/MyDrive/EnerVision_AI/Models/model_results.csv'

def save_model_result(model_name, mae, rmse, r2=None, path=RESULTS_PATH):
    """Append/update this model's metrics in the shared comparison file."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    row = pd.DataFrame([{'model': model_name, 'mae': mae, 'rmse': rmse, 'r2': r2}])

    if os.path.exists(path):
        existing = pd.read_csv(path)
        existing = existing[existing['model'] != model_name]
        df_out = pd.concat([existing, row], ignore_index=True)
    else:
        df_out = row

    df_out.to_csv(path, index=False)
    print(f"✅ Saved metrics for '{model_name}' to:\n{path}")
    print(df_out)

save_model_result('PatchTSMixer', mae, rmse, r2)


## 📈 Step 9: Plot Actual vs Predicted Values


In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(
actuals_flat[:100, 0],
label='Actual',
color='blue',
alpha=0.8
)

plt.plot(
preds_flat[:100, 0],
label='Predicted',
color='red',
linestyle='--',
alpha=0.8
)

plt.title('Comparison between Actual and Predicted Demand using PatchTSMixer (Hourly)')
plt.xlabel('Hours')
plt.ylabel('Demand (kW)')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()


## 💾 Step 10: Save the Trained PatchTSMixer Model Weights


In [ ]:
import joblib

model_save_path = (
    '/content/drive/MyDrive/EnerVision_AI/Models/PatchTSMixer/patchtsmixer_model.pt'
)

torch.save(
    model.state_dict(),
    model_save_path
)

print(
    f"💾 Model weights saved successfully to Google Drive:\n"
    f"{model_save_path}"
)

scaler_save_path = (
    '/content/drive/MyDrive/EnerVision_AI/Models/PatchTSMixer/patchtsmixer_scaler.joblib'
)

joblib.dump(scaler, scaler_save_path)

print(
    f"💾 Scaler saved successfully to Google Drive:\n"
    f"{scaler_save_path}"
)
